### Haar average

average of a unitary matrix is zero:

$$
    \bar{u} = 0
$$

In [ ]:
import numpy as np
from quante.generate.matrix.random import random_matrix
from tqdm import tqdm

def test_haar_average_1(iternum):
    res = np.zeros((2, 2), dtype=complex)
    for i in tqdm(range(iternum)):
        O = random_matrix(2, mtype='CUE')
        res += O
    res /= iternum
    return res
55
test_haar_average_1(10000)

average of a unitary matrix tensor product with its conjugate is proportional to the identity projector:
$$
    \overline{u^* \otimes u} = \frac{1}{d} \sum_{ij} |I\rangle \langle I|
$$
where $\left| I \right\rangle$ is the vectorized identity matrix.

In [ ]:
import numpy as np
from quante.generate.matrix.random import random_matrix
from tqdm import tqdm


def test_haar_average_2(iternum):
    res = np.zeros((4, 4), dtype=complex)
    for i in tqdm(range(iternum)):
        O = random_matrix(2, mtype='CUE')
        res += np.kron(O.conj(), O)
    res /= iternum

    I = np.eye(2, dtype=complex)
    ketI = I.reshape(-1, 1)
    avemat_exact = (1/2) * ketI @ ketI.conj().T
    return res, avemat_exact

avemat, avemat_exact = test_haar_average_2(10000)
print(avemat)
print(avemat_exact)

use notation,
$$
    \mathcal{P}_{A} = |A\rangle \langle A| 
$$
If $A$ act on a d-dimensional Hilbert space, then $\mathcal{P}_{A}$ is a $d^2$ by $d^2$ matrix.

now we consider a 2-site gate matrix with local conservation, i.e., block diagonal matrix with each block being a random unitary matrix drawn from CUE ensemble. For example, 
$$
    U = \begin{pmatrix}
        U_1 & 0 & 0 \\
        0 & U_2 & 0 \\
        0 & 0 & U_3
    \end{pmatrix}
$$
where $U_1$ and $U_3$ is a $1$ by $1$ CUE matrix, $U_2$ is a $2$ by $2$ CUE matrix.

The basis is $$|00\rangle, |01\rangle, |10\rangle, |11\rangle$$

Or equivalently, we can write
$$
    U = \sum_{Q} U_Q
$$
where $Q = 0, \cdots , 2(q - 1)$ is the conserved charge with local dimension $q$.
The dimension of each block is given by $d_Q = q - |Q + 1 - q|$

For $q = 2$, i.e., spin 1/2 system, we have $Q = 0,1,2$ and $d_0 = d_2 = 1, d_1 = 2$.

Now, we want to calculate the Haar average of $U^* \otimes U$,
$$
    \overline{U^* \otimes U} = \sum_{Q_1, Q_2} \overline{U_{Q_1}^* \otimes U_{Q_2}} = \sum_{Q} \overline{U_{Q}^* \otimes U_{Q}}
$$
since the different blocks are independent, the average of the cross terms are zero.

Define the projector on the subspace with charge $Q$ as $P_Q$.

For $q = 2$, i.e., spin 1/2 system, we have
$$
    P_0 = |00\rangle \langle 00| = \begin{pmatrix}
        1 & 0 & 0 & 0 \\
        0 & 0 & 0 & 0 \\
        0 & 0 & 0 & 0 \\
        0 & 0 & 0 & 0 \\
    \end{pmatrix}, \quad
$$
$$
    P_1 = |10\rangle \langle 10| + | 01 \rangle\langle 01 | = \begin{pmatrix}
        0 & 0 & 0 & 0 \\
        0 & 1 & 0 & 0 \\
        0 & 0 & 1 & 0 \\
        0 & 0 & 0 & 0 \\
    \end{pmatrix}, \quad
$$
$$
    P_2 = |11\rangle \langle 11| = \begin{pmatrix}
        0 & 0 & 0 & 0 \\
        0 & 0 & 0 & 0 \\
        0 & 0 & 0 & 0 \\
        0 & 0 & 0 & 1 \\
    \end{pmatrix}, \quad
$$

Then the average is given by
$$
    \overline{U^* \otimes U} = \sum_{Q} \overline{U_{Q}^* \otimes U_{Q}} = \sum_{Q} \frac{1}{d_Q} |P_Q\rangle \langle P_Q|
$$

In [ ]:
import numpy as np
from quante.generate.matrix.random import random_twosite_conserve
from tqdm import tqdm


def test_haar_average_3(iternum):
    res = np.zeros((16, 16), dtype=complex)
    for i in tqdm(range(iternum)):
        O = random_twosite_conserve(2)
        res += np.kron(O.conj(), O)
    res /= iternum

    P0 = np.zeros((4, 4), dtype=complex)
    P0[0, 0] = 1
    P0 = P0.reshape(-1, 1)
    P1 = np.zeros((4, 4), dtype=complex)
    P1[1, 1] = 1
    P1[2, 2] = 1
    P1 = P1.reshape(-1, 1)
    P2 = np.zeros((4, 4), dtype=complex)
    P2[3, 3] = 1
    P2 = P2.reshape(-1, 1)
    avemat_exact = np.zeros((16, 16), dtype=complex)
    avemat_exact += P0 @ P0.conj().T
    avemat_exact += (1/2) * P1 @ P1.conj().T
    avemat_exact += P2 @ P2.conj().T
    return res, avemat_exact

avemat, avemat_exact = test_haar_average_3(10000)
print(avemat.round(1))
print(avemat_exact)

Note that there are only six non-zero elements in the above matrix.

Since the $\overline{U^* \otimes U}$ is of the basis,
$$
    |00,00\rangle, |00,01\rangle, |00,10\rangle, |00,11\rangle, \cdots , |11,00\rangle, |11,01\rangle, |11,10\rangle, |11,11\rangle
$$


Writing them in the basis,
$$
    \overline{U^* \otimes U} = \sum_{Q} \frac{1}{d_Q} |P_Q\rangle \langle P_Q|
    = \frac{1}{1} |00,00\rangle \langle 00,00| + \frac{1}{2} (|01,01\rangle + |10,10\rangle)(\langle 01,01| + \langle 10,10|) + \frac{1}{1} |11,11\rangle \langle 11,11|
$$
Thus, the effective dimension of the two-site gate is $3$ instead of $2^4 = 16$.

writing in two-site basis,
$$
\begin{align*}
    |P_0\rangle &= |00,00\rangle = |00\rangle_1 |00\rangle_2 \\
    |P_1\rangle &= |01,01\rangle + |10,10\rangle = |00\rangle_1 |11\rangle_2 + |11\rangle_1 |00\rangle_2 \\
    |P_2\rangle &= |11,11\rangle = |11\rangle_1 |11\rangle_2 \\
\end{align*}
$$
we only need two local basis, i.e., $|0\rangle \equiv |00\rangle$ and $|1\rangle \equiv |11\rangle$.
$$
\begin{align*}
    |P_0\rangle &= |0\rangle_1 |0\rangle_2 \\
    |P_1\rangle &= |0\rangle_1 |1\rangle_2 + |1\rangle_1 |0\rangle_2 \\
    |P_2\rangle &= |1\rangle_1 |1\rangle_2 \\
\end{align*}
$$

In [ ]:
def test_haar_average_9():
    P0 = np.zeros((4, 4), dtype=complex)
    P0[0, 0] = 1
    P0 = P0.reshape(-1, 1)
    P1 = np.zeros((4, 4), dtype=complex)
    P1[1, 1] = 1
    P1[2, 2] = 1
    P1 = P1.reshape(-1, 1)
    P2 = np.zeros((4, 4), dtype=complex)
    P2[3, 3] = 1
    P2 = P2.reshape(-1, 1)
    avemat_exact = np.zeros((16, 16), dtype=complex)
    avemat_exact += P0 @ P0.conj().T
    avemat_exact += (1/2) * P1 @ P1.conj().T
    avemat_exact += P2 @ P2.conj().T
    
    basis = np.zeros((2, 4), dtype=complex)
    basis[0, 0b00] = 1.
    basis[1, 0b11] = 1.
    I = np.zeros((2, 2, 16), dtype=complex)
    I[0,0] = np.kron(basis[0], basis[0])
    I[0,1] = np.kron(basis[0], basis[1])
    I[1,0] = np.kron(basis[1], basis[0])
    I[1,1] = np.kron(basis[1], basis[1])
    P10 = I[0,0].reshape(-1, 1)
    P11 = (I[0,1] + I[1,0]).reshape(-1, 1)
    P12 = I[1,1].reshape(-1, 1)
    avemat_exact2 = np.zeros((16, 16), dtype=complex)
    avemat_exact2 += P10 @ P10.conj().T
    avemat_exact2 += (1/2) * P11 @ P11.conj().T
    avemat_exact2 += P12 @ P12.conj().T

    avemat_exact2 = avemat_exact2.reshape(*[2]*8).transpose(0,2,1,3,4,6,5,7).reshape(16,16)
    print(np.linalg.norm(avemat_exact - avemat_exact2))



test_haar_average_9()

In [ ]:
def test_haar_average_10():
    basis = np.zeros((2, 4), dtype=complex)
    basis[0, 0b00] = 1.
    basis[1, 0b11] = 1.
    mscrI = np.zeros((2, 2, 16), dtype=complex)
    mscrI[0,0] = np.kron(basis[0], basis[0])
    mscrI[0,1] = np.kron(basis[0], basis[1])
    mscrI[1,0] = np.kron(basis[1], basis[0])
    mscrI[1,1] = np.kron(basis[1], basis[1])
    P10 = mscrI[0,0].reshape(-1, 1)
    P11 = (mscrI[0,1] + mscrI[1,0]).reshape(-1, 1)
    P12 = mscrI[1,1].reshape(-1, 1)
    avemat_exact2 = np.zeros((16, 16), dtype=complex)
    avemat_exact2 += P10 @ P10.conj().T
    avemat_exact2 += (1/2) * P11 @ P11.conj().T
    avemat_exact2 += P12 @ P12.conj().T

    gate1 = np.zeros((2,2,2,2), dtype=complex)
    for I in range(2):
        for J in range(2):
            for K in range(2):
                for L in range(2):
                    gate1[I,J,K,L] = (
                        mscrI[I, J] @ avemat_exact2 @ mscrI[K, L].conj().T
                    )
    return gate1

def test_haar_average_11():
    """
    |    1  2
    |  ┌─┴──┴─┐
    |  └─┬──┬─┘
    |    3  4
    """
    mscrI = np.eye(2)
    P10 = np.kron(mscrI[0], mscrI[0]).reshape(-1, 1)
    P11 = np.kron(mscrI[0], mscrI[1]).reshape(-1, 1) + np.kron(mscrI[1], mscrI[0]).reshape(-1, 1)
    P12 = np.kron(mscrI[1], mscrI[1]).reshape(-1, 1)
    avemat_exact = np.zeros((4,4), dtype=complex)
    avemat_exact += P10 @ P10.conj().T
    avemat_exact += (1/2) * P11 @ P11.conj().T
    avemat_exact += P12 @ P12.conj().T

    gate1 = np.zeros((2,2,2,2), dtype=complex)
    for I in range(2):
        for J in range(2):
            for K in range(2):
                for L in range(2):
                    gate1[I,J,K,L] = (
                        np.kron(mscrI[I], mscrI[J]) @ avemat_exact @ np.kron(mscrI[K], mscrI[L]).conj().T
                    )
    return gate1

gate1 = test_haar_average_11().reshape(4,4)
gate2 = test_haar_average_11().reshape(4,4)
gate1, gate2


now consider the average over four-fold tensor product space,
$$
    \overline{U^* \otimes U \otimes U^* \otimes U} = \sum_{Q} \overline{U_{Q}^* \otimes U_{Q} \otimes U_{Q}^* \otimes U_{Q}} + \sum_{Q_1 \neq Q_2} (\overline{U_{Q_1}^* \otimes U_{Q_1} \otimes U_{Q_2}^* \otimes U_{Q_2}} + \overline{U_{Q_1}^* \otimes U_{Q_2} \otimes U_{Q_2}^* \otimes U_{Q_1}})
$$


The second term is,
$$
    \overline{U_{Q_1}^* \otimes U_{Q_1} \otimes U_{Q_2}^* \otimes U_{Q_2}} = \frac{1}{d_{Q_1} d_{Q_2}} |P_{Q_1}\rangle \langle P_{Q_1}| \otimes |P_{Q_2}\rangle \langle P_{Q_2}| = \frac{1}{d_{Q_1} d_{Q_2}} (|P_{Q_1}\rangle \otimes |P_{Q_2}\rangle)(\langle P_{Q_1}|\otimes  \langle P_{Q_2}|)
$$

Writing in basis:
$$
   \left| P_{Q_{i}} \right\rangle = \sum_{\alpha \in \mathcal{H}_{Q_{i}}} |\alpha \alpha \rangle
$$
Thus, we can define
$$
   | \mathcal{I}_{Q_{1}Q_{2}}^{ +} \rangle = |P_{Q_1}\rangle \otimes |P_{Q_2}\rangle = \sum_{\alpha \in \mathcal{H}_{Q_1}} \sum_{\beta \in \mathcal{H}_{Q_2}} |\alpha \alpha \beta \beta \rangle
$$

Thus,
$$
    \overline{U_{Q_1}^* \otimes U_{Q_1} \otimes U_{Q_2}^* \otimes U_{Q_2}} = \frac{1}{d_{Q_1} d_{Q_2}} | \mathcal{I}_{Q_{1}Q_{2}}^{ +} \rangle \langle \mathcal{I}_{Q_{1}Q_{2}}^{ +} |
$$

The third term is obtained by swapping the second and fourth qubits in the second term,
$$
    \overline{U_{Q_1}^* \otimes U_{Q_2} \otimes U_{Q_2}^* \otimes U_{Q_1}} = \frac{1}{d_{Q_1} d_{Q_2}} |\mathcal{I}_{Q_{1}Q_{2}}^{-}\rangle \langle \mathcal{I}_{Q_{1}Q_{2}}^{-}|
$$

where,
$$
    |\mathcal{I}_{Q_{1}Q_{2}}^{-}\rangle = \sum_{\alpha \in \mathcal{H}_{Q_1}} \sum_{\beta \in \mathcal{H}_{Q_2}} |\alpha \beta \beta \alpha \rangle
$$

For the first term, applying the Weingarten formula for CUE matrix, we have
$$
    \overline{U_{Q}^* \otimes U_{Q} \otimes U_{Q}^* \otimes U_{Q}} = \frac{1}{d_Q^2 - 1}
    \left[
        (| \mathcal{I}^{ +}_{QQ} \rangle \langle \mathcal{I}^{ +}_{QQ} | + | \mathcal{I}^{ -}_{QQ} \rangle \langle \mathcal{I}^{ -}_{QQ} |) - \frac{1}{d_Q} (| \mathcal{I}^{ +}_{QQ} \rangle \langle \mathcal{I}^{ -}_{QQ} | + | \mathcal{I}^{ -}_{QQ} \rangle \langle \mathcal{I}^{ +}_{QQ} |)
    \right]
$$

for $d_Q = 1$, the above formula is not valid, but simply
$$
    \overline{U_{Q}^* \otimes U_{Q} \otimes U_{Q}^* \otimes U_{Q}} = 1
$$

In [ ]:
import numpy as np
from quante.generate.matrix.random import random_twosite_conserve
from tqdm import tqdm
import matplotlib.pyplot as plt

def test_haar_average_4(iternum):
    res = np.zeros((4**4, 4**4), dtype=complex)
    for i in tqdm(range(iternum)):
        O = random_twosite_conserve(2)
        tmp = np.kron(O.conj(), O)
        res += np.kron(tmp, tmp)
    res /= iternum
    return res

def test_haar_average_5():
    I = np.zeros((2, 3, 3, 4, 4, 4, 4), dtype=complex)
    Qs0, Qs1, Qs2 = [0], [1,2], [3]
    for i, Q1 in enumerate([Qs0, Qs1, Qs2]):
        for j, Q2 in enumerate([Qs0, Qs1, Qs2]):
            for alpha in Q1:
                for beta in Q2:
                    I[0, i, j, alpha, alpha, beta, beta] = 1.
                    I[1, i, j, alpha, beta, beta, alpha] = 1.
    # first term
    res_exact = np.zeros((4**4, 4**4), dtype=complex)
    for Q, dQ in enumerate([1,2,1]):
        if dQ == 1:
            res_exact += I[0, Q, Q].reshape(-1, 1) @ I[0, Q, Q].reshape(1, -1)
        else:
            res_exact += (1/(dQ**2 - 1)) * (
                I[0, Q, Q].reshape(-1, 1) @ I[0, Q, Q].reshape(1, -1) +
                I[1, Q, Q].reshape(-1, 1) @ I[1, Q, Q].reshape(1, -1) -
                (1/dQ) * (
                    I[0, Q, Q].reshape(-1, 1) @ I[1, Q, Q].reshape(1, -1) +
                    I[1, Q, Q].reshape(-1, 1) @ I[0, Q, Q].reshape(1, -1)
                )
            )
    for Q1, dQ1 in enumerate([1,2,1]):
        for Q2, dQ2 in enumerate([1,2,1]):
            if Q1 != Q2:
                res_exact += (1/(dQ1 * dQ2)) * (
                    I[0, Q1, Q2].reshape(-1, 1) @ I[0, Q1, Q2].reshape(1, -1) +
                    I[1, Q1, Q2].reshape(-1, 1) @ I[1, Q1, Q2].reshape(1, -1)
                )
    return res_exact

avemat = test_haar_average_4(10000)
avemat_exact = test_haar_average_5()
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(np.real(avemat), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(np.imag(avemat), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(np.real(avemat_exact), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(np.imag(avemat_exact), vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(np.real(avemat_exact - avemat), cmap='bwr')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(np.imag(avemat_exact - avemat), cmap='bwr')
plt.colorbar()

In [ ]:
num = [10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120, 10240]
res = []
for i in num:
    avemat = test_haar_average_4(i)
    avemat_exact = test_haar_average_5()
    res.append(np.max(np.abs(avemat - avemat_exact)))
plt.plot(num, res, 'o-')
plt.yscale('log')
plt.xscale('log')

Thus, we conclude that
$$
    \overline{U^* \otimes U \otimes U^* \otimes U} = 
    \sum_{s = \pm} \sum_{Q_1 \neq Q_2} \frac{1}{d_{Q_1} d_{Q_2}} | \mathcal{I}_{Q_{1}Q_{2}}^{ s} \rangle \langle \mathcal{I}_{Q_{1}Q_{2}}^{ s} | +
    \sum_{s = \pm} \sum_{Q} \frac{1}{d_Q^2 - 1}
    \left[
        | \mathcal{I}^{ s}_{QQ} \rangle \langle \mathcal{I}^{ s}_{QQ} | - \frac{1}{d_Q} | \mathcal{I}^{ s}_{QQ} \rangle \langle \mathcal{I}^{ - s}_{QQ} |
    \right]
$$

for $q = 2$, it involves states,
$$
    | \mathcal{I}_{00}^{ s} \rangle, | \mathcal{I}_{01}^{ s} \rangle, | \mathcal{I}_{02}^{ s} \rangle, | \mathcal{I}_{10}^{ s} \rangle, | \mathcal{I}_{11}^{ s} \rangle, | \mathcal{I}_{12}^{ s} \rangle, | \mathcal{I}_{20}^{ s} \rangle, | \mathcal{I}_{21}^{ s} \rangle, | \mathcal{I}_{22}^{ s} \rangle
$$
$18$ states in total, instead of $4^4 = 256$.

Now, we try to rewrite them in two site basis one by one.

$$
\begin{align*}
    | \mathcal{I}_{00}^{ +} \rangle &= |00,00,00,00\rangle = |0000\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{01}^{ +} \rangle &= |00,00,01,01\rangle + |00,00,10,10\rangle = |0000\rangle_{1} |0011\rangle_{2} + |0011\rangle_{1} |0000\rangle_{2} \\
        | \mathcal{I}_{02}^{ +} \rangle &= |00,00,11,11\rangle = |0011\rangle_{1} |0011\rangle_{2} \\
    | \mathcal{I}_{10}^{ +} \rangle &= |01,01,00,00\rangle + |10,10,00,00\rangle = |0000\rangle_{1} |1100\rangle_{2} + |1100\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{11}^{ +} \rangle &= |01,01,01,01\rangle + |01,01,10,10\rangle + |10,10,01,01\rangle + |10,10,10,10\rangle \\
         &= |0000\rangle_{1} |1111\rangle_{2} + |0011\rangle_{1} |1100\rangle_{2} + |1100\rangle_{1} |0011\rangle_{2} + |1111\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{12}^{ +} \rangle &= |01,01,11,11\rangle + |10,10,11,11\rangle = |0011\rangle_{1} |1111\rangle_{2} + |1111\rangle_{1} |0011\rangle_{2} \\
    | \mathcal{I}_{20}^{ +} \rangle &= |11,11,00,00\rangle = |1100\rangle_{1} |1100\rangle_{2} \\
    | \mathcal{I}_{21}^{ +} \rangle &= |11,11,01,01\rangle + |11,11,10,10\rangle = |1100\rangle_{1} |1111\rangle_{2} + |1111\rangle_{1} |1100\rangle_{2} \\
    | \mathcal{I}_{22}^{ +} \rangle &= |11,11,11,11\rangle = |1111\rangle_{1} |1111\rangle_{2} \\
\end{align*}
$$
and
$$
\begin{align*}
    | \mathcal{I}_{00}^{ -} \rangle &= |00,00,00,00\rangle = |0000\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{01}^{ -} \rangle &= |00,01,01,00\rangle + |00,10,10,00\rangle = |0000\rangle_{1} |0110\rangle_{2} + |0110\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{02}^{ -} \rangle &= |00,11,11,00\rangle = |0110\rangle_{1} |0110\rangle_{2} \\
    | \mathcal{I}_{10}^{ -} \rangle &= |01,00,00,01\rangle + |10,00,00,10\rangle = |0000\rangle_{1} |1001\rangle_{2} + |1001\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{11}^{ -} \rangle &= |01,01,01,01\rangle + |01,10,10,01\rangle + |10,01,01,10\rangle + |10,10,10,10\rangle \\
         &= |0000\rangle_{1} |1111\rangle_{2} + |0110\rangle_{1} |1001\rangle_{2} + |1001\rangle_{1} |0110\rangle_{2} + |1111\rangle_{1} |0000\rangle_{2} \\
    | \mathcal{I}_{12}^{ -} \rangle &= |01,11,11,01\rangle + |10,11,11,10\rangle = |0110\rangle_{1} |1111\rangle_{2} + |1111\rangle_{1} |0110\rangle_{2} \\
    | \mathcal{I}_{20}^{ -} \rangle &= |11,00,00,11\rangle = |1001\rangle_{1} |1001\rangle_{2} \\
    | \mathcal{I}_{21}^{ -} \rangle &= |11,01,01,11\rangle + |11,10,10,11\rangle = |1001\rangle_{1} |1111\rangle_{2} + |1111\rangle_{1} |1001\rangle_{2} \\
    | \mathcal{I}_{22}^{ -} \rangle &= |11,11,11,11\rangle = |1111\rangle_{1} |1111\rangle_{2} \\
\end{align*}
$$

Redefine,
$$
    |0\rangle = |0000\rangle, |1\rangle = |0011\rangle, |2\rangle = |0110\rangle, |3\rangle = |1001\rangle, |4\rangle = |1100\rangle, |5\rangle = |1111\rangle
$$
$$
\begin{align*}
    | \mathcal{I}_{00}^{ +} \rangle &= |0\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{01}^{ +} \rangle &= |0\rangle_{1} |1\rangle_{2} + |1\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{02}^{ +} \rangle &= |1\rangle_{1} |1\rangle_{2} \\
    | \mathcal{I}_{10}^{ +} \rangle &= |0\rangle_{1} |4\rangle_{2} + |4\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{11}^{ +} \rangle &= |0\rangle_{1} |5\rangle_{2} + |1\rangle_{1} |4\rangle_{2} + |4\rangle_{1} |1\rangle_{2} + |5\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{12}^{ +} \rangle &= |1\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |1\rangle_{2} \\
    | \mathcal{I}_{20}^{ +} \rangle &= |4\rangle_{1} |4\rangle_{2} \\
    | \mathcal{I}_{21}^{ +} \rangle &= |4\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |4\rangle_{2} \\
    | \mathcal{I}_{22}^{ +} \rangle &= |5\rangle_{1} |5\rangle_{2} \\
\end{align*}
$$
and
$$
\begin{align*}
    | \mathcal{I}_{00}^{ -} \rangle &= |0\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{01}^{ -} \rangle &= |0\rangle_{1} |2\rangle_{2} + |2\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{02}^{ -} \rangle &= |2\rangle_{1} |2\rangle_{2} \\
    | \mathcal{I}_{10}^{ -} \rangle &= |0\rangle_{1} |3\rangle_{2} + |3\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{11}^{ -} \rangle &= |0\rangle_{1} |5\rangle_{2} + |2\rangle_{1} |3\rangle_{2} + |3\rangle_{1} |2\rangle_{2} + |5\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{12}^{ -} \rangle &= |2\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |2\rangle_{2} \\
    | \mathcal{I}_{20}^{ -} \rangle &= |3\rangle_{1} |3\rangle_{2} \\
    | \mathcal{I}_{21}^{ -} \rangle &= |3\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |3\rangle_{2} \\
    | \mathcal{I}_{22}^{ -} \rangle &= |5\rangle_{1} |5\rangle_{2} \\
\end{align*}
$$

$$
    \overline{U^* \otimes U \otimes U^* \otimes U} = 
    \sum_{s = \pm} \sum_{Q_1 \neq Q_2} \frac{1}{d_{Q_1} d_{Q_2}} | \mathcal{I}_{Q_{1}Q_{2}}^{ s} \rangle \langle \mathcal{I}_{Q_{1}Q_{2}}^{ s} | +
    \sum_{s = \pm} \sum_{Q} \frac{1}{d_Q^2 - 1}
    \left[
        | \mathcal{I}^{ s}_{QQ} \rangle \langle \mathcal{I}^{ s}_{QQ} | - \frac{1}{d_Q} | \mathcal{I}^{ s}_{QQ} \rangle \langle \mathcal{I}^{ - s}_{QQ} |
    \right]
$$

In [ ]:
def test_haar_average_6():
    basis = np.zeros((6, 2**4), dtype=complex)
    basis[0, 0b0000] = 1
    basis[1, 0b0011] = 1
    basis[2, 0b0110] = 1
    basis[3, 0b1001] = 1
    basis[4, 0b1100] = 1
    basis[5, 0b1111] = 1
    I = np.zeros((2, 3, 3, 4**4), dtype=complex)
    I[0,0,0,:] = np.kron(basis[0], basis[0])
    I[0,0,1,:] = np.kron(basis[0], basis[1]) + np.kron(basis[1], basis[0])
    I[0,0,2,:] = np.kron(basis[1], basis[1])
    I[0,1,0,:] = np.kron(basis[0], basis[4]) + np.kron(basis[4], basis[0])
    I[0,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[1], basis[4]) + np.kron(basis[4], basis[1]) + np.kron(basis[5], basis[0])
    I[0,1,2,:] = np.kron(basis[1], basis[5]) + np.kron(basis[5], basis[1])
    I[0,2,0,:] = np.kron(basis[4], basis[4])
    I[0,2,1,:] = np.kron(basis[4], basis[5]) + np.kron(basis[5], basis[4])
    I[0,2,2,:] = np.kron(basis[5], basis[5])
    I[1,0,0,:] = np.kron(basis[0], basis[0])
    I[1,0,1,:] = np.kron(basis[0], basis[2]) + np.kron(basis[2], basis[0])
    I[1,0,2,:] = np.kron(basis[2], basis[2])
    I[1,1,0,:] = np.kron(basis[0], basis[3]) + np.kron(basis[3], basis[0])
    I[1,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[2], basis[3]) + np.kron(basis[3], basis[2]) + np.kron(basis[5], basis[0])
    I[1,1,2,:] = np.kron(basis[2], basis[5]) + np.kron(basis[5], basis[2])
    I[1,2,0,:] = np.kron(basis[3], basis[3])
    I[1,2,1,:] = np.kron(basis[3], basis[5]) + np.kron(basis[5], basis[3])
    I[1,2,2,:] = np.kron(basis[5], basis[5])
    
    res_exact = np.zeros((4**4, 4**4), dtype=complex)
    for s in [0,1]:
        for Q, dQ in enumerate([1,2,1]):
            if dQ == 1:
                res_exact += I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1)/2
            else:
                res_exact += (1/(dQ**2 - 1)) * (
                    I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1) -
                    (1/dQ) * (I[s, Q, Q].reshape(-1, 1) @ I[1-s, Q, Q].reshape(1, -1))
                )
        for Q1, dQ1 in enumerate([1,2,1]):
            for Q2, dQ2 in enumerate([1,2,1]):
                if Q1 != Q2:
                    res_exact += (1/(dQ1 * dQ2)) * (I[s, Q1, Q2].reshape(-1, 1) @ I[s, Q1, Q2].reshape(1, -1))
    return res_exact, basis

avemat_exact, basis = test_haar_average_6()
avemat_exact2_1 = avemat_exact.reshape(*[2]*16).transpose(0,4,1,5,2,6,3,7,8,12,9,13,10,14,11,15).reshape(4**4, 4**4)
avemat_exact1 = test_haar_average_5()
np.linalg.norm(avemat_exact2_1- avemat_exact1)
# avemat_exact2

In [ ]:
def test_haar_average_7(avemat, basis):
    dim = basis.shape[0]
    gate = np.zeros((dim,dim,dim,dim), dtype=complex)
    for I in range(dim):
        for J in range(dim):
            for K in range(dim):
                for L in range(dim):
                    gate[I, J, K, L] = (
                        np.kron(basis[I],basis[J]) @ avemat @ np.kron(basis[K], basis[L]).conj().T
                    )
    return gate
avemat_exact, basis = test_haar_average_6()
gate1 = test_haar_average_7(avemat_exact, basis)
gate1 = gate1.reshape(36,36)

In [ ]:
def test_haar_average_8():
    """
    |    1  2
    |  ┌─┴──┴─┐
    |  └─┬──┬─┘
    |    3  4
    """
    basis = np.eye(6)
    I = np.zeros((2, 3, 3, 36))
    I[0,0,0,:] = np.kron(basis[0], basis[0])
    I[0,0,1,:] = np.kron(basis[0], basis[1]) + np.kron(basis[1], basis[0])
    I[0,0,2,:] = np.kron(basis[1], basis[1])
    I[0,1,0,:] = np.kron(basis[0], basis[4]) + np.kron(basis[4], basis[0])
    I[0,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[1], basis[4]) + np.kron(basis[4], basis[1]) + np.kron(basis[5], basis[0])
    I[0,1,2,:] = np.kron(basis[1], basis[5]) + np.kron(basis[5], basis[1])
    I[0,2,0,:] = np.kron(basis[4], basis[4])
    I[0,2,1,:] = np.kron(basis[4], basis[5]) + np.kron(basis[5], basis[4])
    I[0,2,2,:] = np.kron(basis[5], basis[5])
    I[1,0,0,:] = np.kron(basis[0], basis[0])
    I[1,0,1,:] = np.kron(basis[0], basis[2]) + np.kron(basis[2], basis[0])
    I[1,0,2,:] = np.kron(basis[2], basis[2])
    I[1,1,0,:] = np.kron(basis[0], basis[3]) + np.kron(basis[3], basis[0])
    I[1,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[2], basis[3]) + np.kron(basis[3], basis[2]) + np.kron(basis[5], basis[0])
    I[1,1,2,:] = np.kron(basis[2], basis[5]) + np.kron(basis[5], basis[2])
    I[1,2,0,:] = np.kron(basis[3], basis[3])
    I[1,2,1,:] = np.kron(basis[3], basis[5]) + np.kron(basis[5], basis[3])
    I[1,2,2,:] = np.kron(basis[5], basis[5])

    res = np.zeros((36, 36), dtype=complex)
    for s in [0,1]:
        for Q, dQ in enumerate([1,2,1]):
            if dQ == 1:
                res += I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1)/2
            else:
                res += (1/(dQ**2 - 1)) * (
                    I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1) -
                    (1/dQ) * (I[s, Q, Q].reshape(-1, 1) @ I[1-s, Q, Q].reshape(1, -1))
                )
        for Q1, dQ1 in enumerate([1,2,1]):
            for Q2, dQ2 in enumerate([1,2,1]):
                if Q1 != Q2:
                    res += (1/(dQ1 * dQ2)) * (I[s, Q1, Q2].reshape(-1, 1) @ I[s, Q1, Q2].reshape(1, -1))
    return res

gate2 = test_haar_average_8()
gate1, gate2

In [ ]:
np.linalg.norm(gate1 - gate2)

In [ ]:
len(np.nonzero(gate2)[0])

下面是文章中的顺序：
$$
    |0\rangle = |0000\rangle, |1\rangle = |0011\rangle, |4\rangle = |0110\rangle, |3\rangle = |1001\rangle, |2\rangle = |1100\rangle, |5\rangle = |1111\rangle
$$
$$
\begin{align*}
    | \mathcal{I}_{00}^{ +} \rangle &= |0\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{01}^{ +} \rangle &= |0\rangle_{1} |1\rangle_{2} + |1\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{02}^{ +} \rangle &= |1\rangle_{1} |1\rangle_{2} \\
    | \mathcal{I}_{10}^{ +} \rangle &= |0\rangle_{1} |2\rangle_{2} + |2\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{11}^{ +} \rangle &= |0\rangle_{1} |5\rangle_{2} + |1\rangle_{1} |2\rangle_{2} + |2\rangle_{1} |1\rangle_{2} + |5\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{12}^{ +} \rangle &= |1\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |1\rangle_{2} \\
    | \mathcal{I}_{20}^{ +} \rangle &= |2\rangle_{1} |2\rangle_{2} \\
    | \mathcal{I}_{21}^{ +} \rangle &= |2\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |2\rangle_{2} \\
    | \mathcal{I}_{22}^{ +} \rangle &= |5\rangle_{1} |5\rangle_{2} \\
\end{align*}
$$
and
$$
\begin{align*}
    | \mathcal{I}_{00}^{ -} \rangle &= |0\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{01}^{ -} \rangle &= |0\rangle_{1} |4\rangle_{2} + |4\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{02}^{ -} \rangle &= |4\rangle_{1} |4\rangle_{2} \\
    | \mathcal{I}_{10}^{ -} \rangle &= |0\rangle_{1} |3\rangle_{2} + |3\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{11}^{ -} \rangle &= |0\rangle_{1} |5\rangle_{2} + |4\rangle_{1} |3\rangle_{2} + |3\rangle_{1} |4\rangle_{2} + |5\rangle_{1} |0\rangle_{2} \\
    | \mathcal{I}_{12}^{ -} \rangle &= |4\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |4\rangle_{2} \\
    | \mathcal{I}_{20}^{ -} \rangle &= |3\rangle_{1} |3\rangle_{2} \\
    | \mathcal{I}_{21}^{ -} \rangle &= |3\rangle_{1} |5\rangle_{2} + |5\rangle_{1} |3\rangle_{2} \\
    | \mathcal{I}_{22}^{ -} \rangle &= |5\rangle_{1} |5\rangle_{2} \\
\end{align*}
$$

按照文章的设定顺序，计算 OTOC

In [ ]:
import numpy as np
import quante as qt
device = 'cpu'

def haar_average_1():
    """
    |    1  2
    |  ┌─┴──┴─┐
    |  └─┬──┬─┘
    |    3  4
    """
    basis = np.eye(6)
    I = np.zeros((2, 3, 3, 36))
    I[0,0,0,:] = np.kron(basis[0], basis[0])
    I[0,0,1,:] = np.kron(basis[0], basis[1]) + np.kron(basis[1], basis[0])
    I[0,0,2,:] = np.kron(basis[1], basis[1])
    I[0,1,0,:] = np.kron(basis[0], basis[2]) + np.kron(basis[2], basis[0])
    I[0,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[1], basis[2]) + np.kron(basis[2], basis[1]) + np.kron(basis[5], basis[0])
    I[0,1,2,:] = np.kron(basis[1], basis[5]) + np.kron(basis[5], basis[1])
    I[0,2,0,:] = np.kron(basis[2], basis[2])
    I[0,2,1,:] = np.kron(basis[2], basis[5]) + np.kron(basis[5], basis[2])
    I[0,2,2,:] = np.kron(basis[5], basis[5])
    I[1,0,0,:] = np.kron(basis[0], basis[0])
    I[1,0,1,:] = np.kron(basis[0], basis[4]) + np.kron(basis[4], basis[0])
    I[1,0,2,:] = np.kron(basis[4], basis[4])
    I[1,1,0,:] = np.kron(basis[0], basis[3]) + np.kron(basis[3], basis[0])
    I[1,1,1,:] = np.kron(basis[0], basis[5]) + np.kron(basis[4], basis[3]) + np.kron(basis[3], basis[4]) + np.kron(basis[5], basis[0])
    I[1,1,2,:] = np.kron(basis[4], basis[5]) + np.kron(basis[5], basis[4])
    I[1,2,0,:] = np.kron(basis[3], basis[3])
    I[1,2,1,:] = np.kron(basis[3], basis[5]) + np.kron(basis[5], basis[3])
    I[1,2,2,:] = np.kron(basis[5], basis[5])

    res = np.zeros((36, 36), dtype=float)
    for s in [0,1]:
        for Q, dQ in enumerate([1,2,1]):
            if dQ == 1:
                res += I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1)/2
            else:
                res += (1/(dQ**2 - 1)) * (
                    I[s, Q, Q].reshape(-1, 1) @ I[s, Q, Q].reshape(1, -1) -
                    (1/dQ) * (I[s, Q, Q].reshape(-1, 1) @ I[1-s, Q, Q].reshape(1, -1))
                )
        for Q1, dQ1 in enumerate([1,2,1]):
            for Q2, dQ2 in enumerate([1,2,1]):
                if Q1 != Q2:
                    res += (1/(dQ1 * dQ2)) * (I[s, Q1, Q2].reshape(-1, 1) @ I[s, Q1, Q2].reshape(1, -1))
    return res

In [ ]:
def indx2str(indx):
    a = indx // 6
    b = indx % 6
    return f'{a}{b}'
mat = haar_average_1()
for i in range(36):
    print(f'{indx2str(i)}', end=' => ')
    for j in range(36):
        if abs(mat[i,j]) > 1e-10:
            print(f'{indx2str(j)}', end=', ')
    print()

考虑 Purity 的计算

初态取为：
$$
    \vert \psi \rangle = \mathrm{e}^{ - \mathrm{i} Y \theta / 2} \vert 0 \rangle = \cos(\theta/2) \vert 0 \rangle + \sin(\theta/2) \vert 1 \rangle
$$

所以 $\vert \psi \rangle^{\otimes 4}$ 在上面定义的 basis 下的投影为
$$
    \cos^4(\theta/2) |0\rangle + \sin^4(\theta/2) |5\rangle + \cos^2(\theta/2)\sin^2(\theta/2)(|1\rangle + |2\rangle + |3\rangle + |4\rangle)
$$

而上边界则根据是否是子系统来决定，分成两种情况。

```text
╭-╮ ╭-╮
| | | |
```
也就是：
$$
    \vert \psi \rangle = \sum_{i,j=0}^{5} |i\rangle_{1}|i\rangle_{2}|j\rangle_{3}|j\rangle_{4}
$$
投影到上面的 basis 下，得到
$$
    \vert 0 \rangle + \vert 1 \rangle + \vert 2 \rangle + \vert 5 \rangle 
$$


还有一种情况是
```text
╭-----╮
| ╭-╮ |
| | | |
```
也就是:
$$
    \vert \psi \rangle = \sum_{i,j=0}^{5} |i\rangle_{1}|j\rangle_{2}|j\rangle_{3}|i\rangle_{4}
$$
投影到上面的 basis 下，得到
$$
    \vert 0 \rangle + \vert 3 \rangle + \vert 4 \rangle + \vert 5 \rangle
$$


projector
$$
    \rho_{A,Q} = \sum_{s = 1}^{N_A} \Pi_{s} \rho_{A} \Pi_{s} = \frac{1}{N_{A}+1} \sum_{j = 0}^{N_{A}} \mathrm{e}^{ - 2\pi \mathrm{i} jQ_{A}/(N_{A} + 1)} \rho_{A} \mathrm{e}^{2\pi\mathrm{i} jQ_{A}/(N_{A} + 1)} 
$$

In [ ]:
import quante as qt
import numpy as np
from quante.generate.basis.spin_half.bitsoperation import count_tot_down
op = qt.generate.operas
NA = 4
rhoA = qt.generate.matrix.random_matrix(dim=2**NA)

Pi = []
for s in range(NA+1):
    Pis = np.zeros((2**NA, 2**NA))
    for i in range(2**NA):
        if count_tot_down(i) == s:
            Pis[i,i] = 1.
    Pi.append(Pis)
rhoAQ = sum(Pis @ rhoA @ Pis for Pis in Pi)  # block diagonalized rhoA

QAs = []
for j in range(NA+1):
    QA = np.zeros((2**NA, 2**NA), dtype=complex)
    for i in range(2**NA):
        QA[i,i] = np.exp(-2*np.pi*1j*j*count_tot_down(i)/(NA+1))
    QAs.append(QA)
rhoAQ2 = sum(QA @ rhoA @ QA.conj().T for QA in QAs)/(NA+1)  # block diagonalized rhoA
np.real_if_close(rhoAQ - rhoAQ2)


这是可以用傅里叶变换证明

定义
$$
    P_{j} = \frac{1}{ \sqrt{N_A + 1}} \sum_{s = 0}^{N_A} \mathrm{e}^{ - 2\pi \mathrm{i} j s / (N_A + 1)} \Pi_{s}
$$

因而
$$
    \Pi_{s} = \frac{1}{ \sqrt{N_A + 1}} \sum_{j = 0}^{N_A} \mathrm{e}^{ 2\pi \mathrm{i} j s / (N_A + 1)} P_{j}
$$

同时注意到：
$$
    Q_{A} = \sum_{s = 0}^{N_A} s \Pi_{s}
$$

所以&nbsp;${ P_{s} }$ 实际上可以写出是：
$$
    P_{j} = \frac{1}{ \sqrt{N_A + 1}} \mathrm{e}^{ - 2\pi \mathrm{i} j Q_{A} / (N_A + 1)} 
$$

因此投影算法实际上就是：
$$
    \Pi_{s} = \frac{1}{N_A + 1} \sum_{j = 0}^{N_A} \mathrm{e}^{ 2\pi \mathrm{i} j s / (N_A + 1)} \mathrm{e}^{ - 2\pi \mathrm{i} j Q_{A} / (N_A + 1)}
$$

将这一结果带入，
$$
\begin{align*}
    \rho_{A,Q} &= \sum_{s = 1}^{N_A} \Pi_{s} \rho_{A} \Pi_{s} \\
    &= \frac{1}{N_{A}+1} \sum_{j = 0}^{N_{A}}  \mathrm{e}^{ - 2\pi \mathrm{i} j Q_{A}/(N_{A} + 1)} \rho_{A} \mathrm{e}^{2\pi\mathrm{i} j Q_{A}/(N_{A} + 1)}
\end{align*}
$$
也就完成了证明

所以实际上就是需要表示出：
$$
    \mathrm{e}^{ - 2\pi \mathrm{i} j Q_{A}/(N_{A} + 1)} = \prod_{i = 1}^{N_A} \mathrm{e}^{ - 2\pi \mathrm{i} j n_{i}/(N_{A} + 1)}
$$


对于文中的表示，local的态就是：
$$
    \vert 0 \rangle + \mathrm{e}^{2\pi \mathrm{i} j / (N_A + 1)} \vert 3 \rangle + \mathrm{e}^{ - 2\pi \mathrm{i} j / (N_A + 1)} \vert 4 \rangle + \vert 5 \rangle 
$$

In [ ]:
import quante.bridge.torch_utils as qtc
import torch as tc
import quante as qt
import numpy as np

L = 100
NA = 16
cutoff = 1e-8
usevec = False  # full vector or MPS
device = 'cpu'

gate = qt.generate.matrix.rc.u1m2()
res = {}
for th in [0.4, 0.6, 0.8]:
    theta = th
    costh2 = np.cos(theta/2)**2
    sinth2 = np.sin(theta/2)**2
    state_site = np.array(
        [costh2*costh2,costh2*sinth2,costh2*sinth2,costh2*sinth2,costh2*sinth2,sinth2*sinth2]
    )
    f_up = np.array([
        1., 1., 1., 0., 0., 1.,
    ])
    f_down = np.array([
        1., 0., 0., 1., 1., 1.,
    ])
    state_mps = qtc.MPS.from_product_state2([state_site]*L, dtype=tc.float64, device=device)
    bnd_purity_mps = qtc.MPS.from_product_state2([f_down]*NA + [f_up]*(L-NA), dtype=tc.float64, device=device)
    pur_as = []
    pur_aqs = []
    ts = []
    # from quante.bridge.torch_utils.linalg import eigh
    # import dowhen
    # dowhen.do("print(mat.shape)").when(eigh, "S, V = tc.linalg.eigh(rho)")
    # with Timer(eigh):
    for t in range(200):
        pur_a = -np.log(np.real(bnd_purity_mps.inner(state_mps).item()))
        if usevec:
            state = state_mps.to_vector().numpy()
            bnd_purity = bnd_purity_mps.to_vector().numpy()
            pur_a = - np.log(np.real(np.vdot(bnd_purity, state)))
        val = 0.
        for k in range(NA+1):
            kk = 2*np.pi*k/(NA+1)
            f_down_k = np.array([1, 0, 0, np.exp(1j*kk), np.exp(-1j*kk), 1])
            bnd_mps = qtc.MPS.from_product_state2([f_down_k]*NA + [f_up]*(L-NA), dtype=tc.complex128, device=device)
            if usevec:
                bnd = bnd_mps.to_vector().numpy()
                val += np.vdot(bnd, state)/(NA+1)
            else:
                val += bnd_mps.inner(state_mps).item()/(NA+1)

        pur_aq = - np.log(np.real(val))
        pur_as.append(pur_a)
        pur_aqs.append(pur_aq)
        ts .append(t)
        print('t:', t, end=' ')
        print('purity a:', pur_a, end=' ')
        print('purity aq:', pur_aq, end=' ')
        print('maxbonddim:', state_mps.maxbonddim())
        res[f'theta={th}'] = {'ts': ts, 'pur_as': pur_as, 'pur_aqs': pur_aqs}
        tc.cuda.empty_cache()
        gate = gate.reshape(6,6,6,6)
        for i in range(1,L-1,2):
            state_mps.apply_gate_(i, gate, trunc_para=(1300, None, cutoff), normalize=True)
        for i in range(0,L-1,2):
            state_mps.apply_gate_(i, gate, trunc_para=(1300, None, cutoff), normalize=True)

In [ ]:
import matplotlib.pyplot as plt
import quante as qt
qt.basicfun.plt_style_use()
import matplotlib as mpl
import numpy as np
th = 0.4
for th in [0.4, 0.6, 0.8]:
    data = res[f"theta={th}"]
    cm = mpl.cm.ScalarMappable(cmap='Reds_r',norm=mpl.colors.Normalize(vmin=0.4,vmax=np.pi/2))
    plt.plot([i+1 for i in data['ts']], [aq-a for a,aq in zip(data['pur_as'], data['pur_aqs'])],
            lw=2,c=cm.to_rgba(th),mec='k',ms=0,marker='h',label=r'$\theta$'+f'$={th}$')